# DFU-ImageGuard — Q1 Post-hoc Corrections (No Retraining)

Regenerates threshold-safe metrics, duplicate-group bootstrap intervals, cluster-aware comparisons, threshold-aware uncertainty, risk–coverage curves, and Q1 claim safeguards from saved OOF predictions. It does **not** train or modify model weights.


In [ ]:
import sys, subprocess, shutil, importlib, json
from pathlib import Path

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
    "numpy>=1.26", "pandas>=2.2", "scipy>=1.13", "scikit-learn>=1.5",
    "matplotlib>=3.9", "tabulate>=0.9"
])
REPO = Path("/content/DFU-ImageGuard")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.check_call(["git", "clone", "https://github.com/AzizulHakim00/DFU-ImageGuard.git", str(REPO)])
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."): del sys.modules[module_name]
importlib.invalidate_caches()
if str(REPO) not in sys.path: sys.path.insert(0, str(REPO))
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as exc: print(f"Drive mount note: {exc}")
from src.q1_posthoc import run_q1_posthoc_corrections
RUN_ID = None  # Example: "20260801_053247"
CANDIDATE_ROOTS = [Path("/content/drive/MyDrive/DFU-ImageGuard/runs"), Path("/content/DFU-ImageGuard-local/runs")]
def completed_runs():
    return [p for base in CANDIDATE_ROOTS if base.exists() for p in base.iterdir() if p.is_dir() and (p / "predictions/all_oof_predictions.csv").exists()]
if RUN_ID:
    matches=[base/RUN_ID for base in CANDIDATE_ROOTS if (base/RUN_ID).exists()]
    if not matches: raise FileNotFoundError(RUN_ID)
    run_root=matches[0]
else:
    candidates=completed_runs()
    if not candidates: raise FileNotFoundError("No completed OOF run found")
    run_root=max(candidates,key=lambda p:p.stat().st_mtime)
print(f"Correcting saved run without retraining: {run_root}")
RESULT=run_q1_posthoc_corrections(run_root, primary_model="DFU-ImageGuard", bootstrap_repetitions=1000, permutation_repetitions=10000, seed=2026, overwrite_primary_tables=True)
from IPython.display import display
m=RESULT["metrics"]
display(m[m.state=="calibrated_fold_specific_thresholds"][["model","accuracy","balanced_accuracy","recall_sensitivity","specificity","f1","mcc","roc_auc","pr_auc","brier_score","ece","fn","fp"]].sort_values("balanced_accuracy",ascending=False))
print(json.dumps(RESULT["readiness"],indent=2))
print(f"Corrected report: {run_root / 'Q1_POSTHOC_REPORT.md'}")
